# Car Detection in Snow – Main Notebook
**Course:** D7047E Advanced Deep Learning | LTU VT2026  
**Dataset:** [Nordic Vehicle Dataset (NVD)](https://nvd.ltu-ai.dev/)  
**Base model:** YOLOv9 via `ultralytics`

---

This is the single entry-point notebook for the project. It is organised into the following sections:

1. [Environment & Imports](#1-environment--imports)
2. [Exploratory Data Analysis (EDA)](#2-exploratory-data-analysis-eda)
3. [Dataset Preparation](#3-dataset-preparation)
4. [Training](#4-training)
   - 4.1 Zero-shot baseline — YOLOv9 (COCO-pretrained, no fine-tuning)
   - 4.2 YOLOv9 fine-tuning
   - 4.3 COCO-format conversion for RT-DETR
   - 4.4 RT-DETR zero-shot baseline
   - 4.5 RT-DETR fine-tuning
   - 4.6 Faster R-CNN fine-tuning
   - 4.7 Faster R-CNN zero-shot baseline
   - 4.8 Results comparison table
5. [Phase 3 — Augmentation Ablation & Hyperparameter Sweep](#phase-3)
   - 4.9 Augmentation preview
   - 4.9A YOLOv9 snow augmentation training
   - 4.9B YOLOv9 full augmentation training
   - 4.10A RT-DETR snow augmentation training
   - 4.10B RT-DETR full augmentation training
   - 4.11A Faster R-CNN snow augmentation training
   - 4.11B Faster R-CNN full augmentation training
   - 4.12 Augmentation ablation results
   - 4.13 Hyperparameter sweep
   - 4.14 Final model training
6. [Evaluation](#5-evaluation)
7. [Error Analysis](#6-error-analysis)


## 1. Environment & Imports

In [ ]:

import os
import random
from pathlib import Path
import shutil

import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm import tqdm

import yaml

# Reproducibility
random.seed(42)
np.random.seed(42)

# Paths
ROOT       = Path(".").resolve()  # project root
DATA_RAW   = ROOT / "data" / "raw"
DATA_PROC  = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"
RESULTS    = ROOT / "results"
FIGURES    = RESULTS / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

print(ROOT)
print(MODELS_DIR)
print("Paths OK")

## 2. Exploratory Data Analysis (EDA)

In [ ]:

# -- EDA - Step 1: Load annotations ------------------------------------------
# NVD uses a FLAT layout: all splits share the same images/ and labels/ dirs.
# Each split loads only the frames listed in its .txt file.

import sys
sys.path.insert(0, str(ROOT / "scripts"))

from data_utils import load_yolo_split_from_txt, compute_stats
from data_utils.reporting import print_stats


CLASS_NAMES = ["car"]

train_split = load_yolo_split_from_txt(DATA_RAW / "train.txt", root=DATA_RAW, split_name="train", class_names=CLASS_NAMES)
val_split   = load_yolo_split_from_txt(DATA_RAW / "val.txt",   root=DATA_RAW, split_name="val",   class_names=CLASS_NAMES)
test_split  = load_yolo_split_from_txt(DATA_RAW / "test.txt",  root=DATA_RAW, split_name="test",  class_names=CLASS_NAMES)

splits = [train_split, val_split, test_split]

for split in splits:
    stats = compute_stats(split)
    print_stats(stats)


In [ ]:

# -- EDA - Annotation quality check ---
# Surfaces potential issues to document in the Methodology section:
#   - Images with no annotations (empty frames)
#   - Crowd boxes (iscrowd=1): groups annotated as one box, not per-instance
#   - Tiny boxes (<32x32 px): often truncated objects at frame edges or label noise
#   - Images where width/height was missing in the annotation file

for split in splits:
    stats = compute_stats(split)
    print(f"[{split.name}]  crowd={stats['crowd_boxes']}  "
          f"tiny={stats['tiny_boxes']}  "
          f"empty_images={stats['images_without_boxes']}  "
          f"missing_dims={stats['missing_dims']}")


In [ ]:

# -- EDA - Step 2: Visualise annotated samples --------------------------------
from eda.visualizer import plot_annotated_samples

fig = plot_annotated_samples(train_split, n=25, cols=5, seed=42)
fig.savefig(FIGURES / "eda_annotated_samples.png", bbox_inches="tight", dpi=120)
plt.show()


In [ ]:

# -- EDA - Step 3: Distribution plots (all four in one dashboard) -------------
from eda.plots import plot_eda_dashboard

train_stats = compute_stats(train_split)

fig = plot_eda_dashboard(train_stats)
fig.savefig(FIGURES / "eda_dashboard_train.png", bbox_inches="tight", dpi=120)
plt.show()


In [ ]:

# -- EDA - Step 4: Bounding-box centre heatmap --------------------------------
from eda.visualizer import plot_bbox_heatmap

fig = plot_bbox_heatmap(train_stats)
fig.savefig(FIGURES / "eda_bbox_heatmap_train.png", bbox_inches="tight", dpi=120)
plt.show()


## 3. Dataset Preparation

No conversion needed – the NVD dataset is already in YOLO format.
`configs/data.yaml` points directly to the raw data and can be passed straight to ultralytics.

Run the cells below once to verify the dataset is correctly set up before training.

In [ ]:

# -- Dataset Preparation - Step 1: Inspect dataset stats ----------------------
# Run this cell to confirm all three splits load correctly from configs/data.yaml.
# Equivalent terminal command:
#   python scripts/inspect_dataset.py --data-yaml configs/data.yaml

import subprocess
import sys

result = subprocess.run(
    [sys.executable, "scripts/inspect_dataset.py",
     "--data-yaml", "configs/data.yaml"],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)


In [ ]:

# -- Dataset Preparation - Step 2: Verify annotations visually ----------------
# Draws bounding boxes on a sample of images and saves them to results/figures/.
# Equivalent terminal command:
#   python scripts/verify_annotations.py --data-yaml configs/data.yaml --split train --n 10 --output results/figures/verify_train/

import subprocess
import sys

result = subprocess.run(
    [sys.executable, "scripts/verify_annotations.py",
     "--data-yaml", "configs/data.yaml",
     "--split", "train",
     "--n", "10",
     "--output", "results/figures/verify_train/"],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)


## 4. Training

All training runs use the `scripts/training/` package (SRP/SoC design):  
- `training.models` — `TrainingConfig`, `EvalMetrics` dataclasses  
- `training.yolo_trainer` — YOLOv9 zero-shot eval + fine-tuning  
- `training.detr_trainer` — RT-DETR fine-tuning + eval  
- `training.frcnn_trainer` — Faster R-CNN fine-tuning + eval  
- `training.evaluator` — vendor result → `EvalMetrics` conversion  
- `training.results_writer` — CSV persistence  

Results accumulate in `results/model_comparison.csv`.

In [ ]:

# --- Phase 2 imports & shared config ----------------------------------------
import sys
from pathlib import Path

sys.path.insert(0, str(ROOT / "scripts"))

from training import (
    TrainingConfig,
    EvalMetrics,
    run_yolo_zero_shot_eval,
    run_yolo_fine_tuning,
    eval_yolo_checkpoint,
    run_detr_zero_shot_eval,
    run_detr_fine_tuning,
    eval_detr_checkpoint,
    run_frcnn_zero_shot_eval,
    run_frcnn_fine_tuning,
    eval_frcnn_checkpoint,
    save_zero_shot_result,
    append_comparison_row,
)

DATA_YAML        = str(ROOT / "configs" / "data.yaml")
COMPARISON_CSV   = str(RESULTS / "model_comparison.csv")
TRAIN_COCO_JSON  = str(DATA_PROC / "annotations" / "instances_train.json")
VAL_COCO_JSON    = str(DATA_PROC / "annotations" / "instances_val.json")
IMAGES_DIR       = str(DATA_RAW / "images")

# Change to "cpu" if no GPU is available
DEVICE = "0"

print("Phase 2 imports OK")


### 4.1 Zero-shot baseline

Run COCO-pretrained YOLOv9 on the NVD val set **without any fine-tuning**.  
This is the "before" anchor that makes fine-tuning gains meaningful.  
Result saved to `results/baseline_zero_shot.csv`.


In [ ]:

# Run once before training anything — sets the "before fine-tuning" anchor.
baseline_metrics = run_yolo_zero_shot_eval(
    model_name="yolov9c.pt",
    data_yaml=DATA_YAML,
    split="val",
    device=DEVICE,
)


append_comparison_row(baseline_metrics, COMPARISON_CSV)

print(f"mAP@0.5     : {baseline_metrics.map50:.4f}")
print(f"mAP@0.5:0.95: {baseline_metrics.map50_95:.4f}")
print(f"Precision   : {baseline_metrics.precision:.4f}")
print(f"Recall      : {baseline_metrics.recall:.4f}")
print(f"FPS         : {baseline_metrics.fps:.1f}")


### 4.2 YOLOv9 fine-tuning

Fine-tune `yolov9c` on the NVD training split.  
- Backbone frozen for the first run (`freeze=10`) — only the detection head adapts.  
- Best checkpoint copied to `models/yolov9_nvd_best.pt`.


In [ ]:



yolo_config = TrainingConfig(
    model_name="yolov9c",
    data_yaml=DATA_YAML,
    epochs=100,
    batch=4,
    imgsz=1024,
    lr0=0.001,
    freeze=None,
    device=DEVICE,
    output_dir = str(MODELS_DIR / "yolov9c"),  # ← THIS WAS MISSING
    run_name="yolov9_nvd",
)

yolo_best_pt = run_yolo_fine_tuning(yolo_config)

# Copy to canonical model path
yolo_dest = MODELS_DIR / "yolov9_nvd_best.pt"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(yolo_best_pt, yolo_dest)
print(f"Best checkpoint → {yolo_dest}")

# Evaluate on val set
yolo_metrics = eval_yolo_checkpoint(
    weights=yolo_dest,
    data_yaml=DATA_YAML,
    split="val",
    device=DEVICE,
    model_label="YOLOv9c (fine-tuned)",
)
append_comparison_row(yolo_metrics, COMPARISON_CSV)

print(f"\nmAP@0.5     : {yolo_metrics.map50:.4f}")
print(f"mAP@0.5:0.95: {yolo_metrics.map50_95:.4f}")
print(f"Precision   : {yolo_metrics.precision:.4f}")
print(f"Recall      : {yolo_metrics.recall:.4f}")
print(f"FPS         : {yolo_metrics.fps:.1f}")


### 4.3 COCO-format conversion for RT DETR

Before running DETR training, generate COCO JSON annotations from the YOLO split files.
This step creates `data/processed/annotations/instances_train.json` and `instances_val.json`.
</VSCode.Cell>
<VSCode.Cell language="python">

In [ ]:

import subprocess
import sys

result = subprocess.run(
    [sys.executable, "scripts/prepare_coco_format.py",
     "--data-yaml", DATA_YAML,
     "--output", str(DATA_PROC)],
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
else:
    print("COCO format annotations generated successfully.")

### 4.4 RT DETR Zero-shot baseline

In [ ]:

# Zero-shot RT-DETR — COCO-pretrained, no fine-tuning
# Prerequisite: COCO JSON annotations must exist (run the prepare_coco_format cell first)
rtdetr_zs_metrics = run_detr_zero_shot_eval(
    val_json=VAL_COCO_JSON,
    images_dir=IMAGES_DIR,
    model_name="PekingU/rtdetr_r50vd",
    split="val",
    device_str=DEVICE,
)


append_comparison_row(rtdetr_zs_metrics, COMPARISON_CSV)

print(f"mAP@0.5     : {rtdetr_zs_metrics.map50:.4f}")
print(f"mAP@0.5:0.95: {rtdetr_zs_metrics.map50_95:.4f}")
print(f"Precision   : {rtdetr_zs_metrics.precision:.4f}")
print(f"Recall      : {rtdetr_zs_metrics.recall:.4f}")
print(f"FPS         : {rtdetr_zs_metrics.fps:.1f}")


### 4.5 RT-DETR fine-tuning

In [ ]:
rtdetr_config = TrainingConfig(
    model_name="PekingU/rtdetr_r50vd",
    data_yaml="",              # RT-DETR uses COCO JSON, not data.yaml
    epochs=100,   # change epoch to 
    batch=4,
    lr0=1e-5,
    freeze=None,                 # freeze all 8 backbone blocks (stem + 4 ResNet stages); mirrors YOLOv9 freeze=10
    device=DEVICE,
    output_dir=str((MODELS_DIR / "rtdetr").resolve()),
    run_name="rtdetr_nvd",
)

rtdetr_best_dir = run_detr_fine_tuning(
    rtdetr_config,
    train_json=TRAIN_COCO_JSON,
    val_json=VAL_COCO_JSON,
    images_dir=IMAGES_DIR,
)
print(f"Best checkpoint → {rtdetr_best_dir}")

rtdetr_metrics = eval_detr_checkpoint(
    checkpoint_dir=rtdetr_best_dir,
    val_json=VAL_COCO_JSON,
    images_dir=IMAGES_DIR,
    split="val",
    device_str=DEVICE,
)
append_comparison_row(rtdetr_metrics, COMPARISON_CSV)

print(f"\nmAP@0.5     : {rtdetr_metrics.map50:.4f}")
print(f"mAP@0.5:0.95: {rtdetr_metrics.map50_95:.4f}")
print(f"Precision   : {rtdetr_metrics.precision:.4f}")
print(f"Recall      : {rtdetr_metrics.recall:.4f}")
print(f"FPS         : {rtdetr_metrics.fps:.1f}")

### 4.6 Faster R-CNN 

Fine-tune `fasterrcnn_resnet50_fpn` (torchvision) on NVD.  
Reads YOLO-format labels directly — no COCO JSON required.  
Gives a classic two-stage detector baseline to compare against.


### Faster R-CNN Zero-shot Baseline


In [ ]:

# Zero-shot Faster R-CNN — COCO-pretrained, no fine-tuning
frcnn_zs_metrics = run_frcnn_zero_shot_eval(
    data_yaml=DATA_YAML,
    split="val",
    device_str=DEVICE,
)

append_comparison_row(frcnn_zs_metrics, COMPARISON_CSV)

print(f"mAP@0.5     : {frcnn_zs_metrics.map50:.4f}")
print(f"mAP@0.5:0.95: {frcnn_zs_metrics.map50_95:.4f}")
print(f"Precision   : {frcnn_zs_metrics.precision:.4f}")
print(f"Recall      : {frcnn_zs_metrics.recall:.4f}")
print(f"FPS         : {frcnn_zs_metrics.fps:.1f}")


### 4.7 Faster R-CNN Fine-tuning


In [ ]:

frcnn_config = TrainingConfig(
    model_name="fasterrcnn_resnet50_fpn",
    data_yaml=DATA_YAML,
    epochs=100,
    batch=4,
    lr0=0.005,
    freeze=None,                 # freeze ResNet body, train FPN + heads
    device=DEVICE,
    output_dir = str(MODELS_DIR / "frcnn"),  # ← THIS WAS MISSING,    # Now uses default "advanced-ai-project"
    run_name="frcnn_nvd",
)

frcnn_best_pt = run_frcnn_fine_tuning(frcnn_config, data_yaml=DATA_YAML)

frcnn_dest = MODELS_DIR / "frcnn_nvd_best.pt"
shutil.copy2(frcnn_best_pt, frcnn_dest)
print(f"Best checkpoint → {frcnn_dest}")

frcnn_metrics = eval_frcnn_checkpoint(
    weights=frcnn_dest,
    data_yaml=DATA_YAML,
    split="val",
    device_str=DEVICE,
)
append_comparison_row(frcnn_metrics, COMPARISON_CSV)

print(f"\nmAP@0.5     : {frcnn_metrics.map50:.4f}")
print(f"mAP@0.5:0.95: {frcnn_metrics.map50_95:.4f}")
print(f"Precision   : {frcnn_metrics.precision:.4f}")
print(f"Recall      : {frcnn_metrics.recall:.4f}")
print(f"FPS         : {frcnn_metrics.fps:.1f}")


### 4.8 Results comparison table

Load `results/model_comparison.csv` and display all models side by side.


In [ ]:

import pandas as pd

comparison_path = Path(COMPARISON_CSV)
if comparison_path.exists():
    df = pd.read_csv(comparison_path)
    # Highlight best value per numeric column
    numeric_cols = ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "FPS"]
    display(
        df.style
          .highlight_max(subset=numeric_cols, color="lightgreen")
          .format({c: "{:.4f}" for c in numeric_cols if c != "FPS"})
          .format({"FPS": "{:.1f}"})
          .set_caption("Phase 2 — Model Comparison (val set)")
    )
else:
    print(f"No comparison CSV yet at {comparison_path}. Run the training cells first.")


---
<a id="phase-3"></a>

## Phase 3 — Augmentation Ablation & Hyperparameter Sweep

Nine training runs compare **3 models × 3 augmentation variants**:

| Section | Model | Augmentation | Run name |
|---------|-------|-------------|----------|
| 4.2  | YOLOv9c | None (ultralytics default) | `yolov9_nvd` |
| 4.9A  | YOLOv9c | Snow | `yolov9_snow` |
| 4.9B  | YOLOv9c | Full | `yolov9_full` |
| 4.5  | RT-DETR | None | `rtdetr_nvd` |
| 4.10A | RT-DETR | Snow | `rtdetr_snow` |
| 4.10B | RT-DETR | Full | `rtdetr_full` |
| 4.7  | Faster R-CNN | None | `frcnn_nvd` |
| 4.11A | Faster R-CNN | Snow | `frcnn_snow` |
| 4.11B | Faster R-CNN | Full | `frcnn_full` |

**Snow pipeline:** `RandomSnow` + `RandomFog` + `RandomBrightnessContrast`  
**Full pipeline:** snow pipeline + `GaussNoise` + `MotionBlur`

All runs use identical architecture, data split, and hyperparameters — augmentation is the only variable.
Pipelines are defined in `scripts/augmentations.py` and injected per model:
- **YOLOv9**: `inject_into_yolo()` callback at `on_train_start`
- **RT-DETR**: `build_detr_pipeline()` in `NVDCocoDataset.__getitem__` (xywh-absolute)
- **Faster R-CNN**: `build_frcnn_pipeline()` in `NVDDetectionDataset.__getitem__` (xyxy-absolute)


### 4.9 Augmentation Preview

Visualise the two augmentation pipelines before training.
Saves `results/figures/aug_preview_snow.png` and `aug_preview_full.png`.


In [ ]:

# ── 4.6 Augmentation preview ─────────────────────────────────────────────────
import sys
sys.path.insert(0, str(ROOT / "scripts"))
from augmentations import build_pipeline, visualize_augmentation

# Load training image paths from data.yaml
with open(DATA_YAML) as f:
    _dcfg = yaml.safe_load(f)

_root = Path(_dcfg.get("path", Path(DATA_YAML).parent))
if not _root.is_absolute():
    _root = Path(DATA_YAML).parent / _root

_train_imgs = [
    (_root / l.strip())
    for l in (_root / _dcfg["train"]).read_text().splitlines()
    if l.strip()
]

snow_pipeline = build_pipeline("snow")
full_pipeline = build_pipeline("full")

fig_snow = visualize_augmentation(_train_imgs, snow_pipeline, n=4, seed=42)
fig_snow.savefig(FIGURES / "aug_preview_snow.png", dpi=120, bbox_inches="tight")
plt.show()

fig_full = visualize_augmentation(_train_imgs, full_pipeline, n=4, seed=42)
fig_full.savefig(FIGURES / "aug_preview_full.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"Saved previews → {FIGURES}")

### 4.9A Snow Augmentation Training (YOLOv9)

Fine-tune YOLOv9c with the **snow pipeline** injected into the training dataloader:
`RandomSnow` + `RandomFog` + `RandomBrightnessContrast`.
Same architecture, same split, same hyperparameters as section 4.2 — augmentation is the only variable.
Checkpoint saved to `models/yolov9_snow_best.pt`.


In [ ]:

# ── 4.9A YOLOv9 snow augmentation training ─────────────────────────────────────
snow_config = TrainingConfig(
    model_name="yolov9c",
    data_yaml=DATA_YAML,
    epochs=100,
    batch=4,
    imgsz=1024,
    lr0=0.001,
    freeze=None,
    device=DEVICE,
    output_dir=str((MODELS_DIR / "yolov9c").resolve()),
    run_name="yolov9_snow",
    project="nvd-car-detection",
)

snow_best_pt = run_yolo_fine_tuning(snow_config, aug_variant="snow")
snow_best_pt_dest = MODELS_DIR / "yolov9_snow_best.pt"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(snow_best_pt, snow_best_pt_dest)
print(f"Snow-aug checkpoint → {snow_best_pt_dest}")

snow_metrics = eval_yolo_checkpoint(
    weights=snow_best_pt_dest,
    data_yaml=DATA_YAML,
    split="val",
    device=DEVICE,
)
append_comparison_row(snow_metrics, COMPARISON_CSV)

print(f"\nmAP@0.5     : {snow_metrics.map50:.4f}")
print(f"mAP@0.5:0.95: {snow_metrics.map50_95:.4f}")
print(f"Precision   : {snow_metrics.precision:.4f}")
print(f"Recall      : {snow_metrics.recall:.4f}")
print(f"FPS         : {snow_metrics.fps:.1f}")


### 4.9B Full Augmentation Training (YOLOv9)

Add `GaussNoise` and `MotionBlur` on top of the snow pipeline.
Checkpoint saved to `models/yolov9_full_best.pt`.


In [ ]:

# ── 4.9B YOLOv9 full augmentation training ─────────────────────────────────────
full_config = TrainingConfig(
    model_name="yolov9c",
    data_yaml=DATA_YAML,
    epochs=100,
    batch=4,
    imgsz=1024,
    lr0=0.001,
    freeze=None,
    device=DEVICE,
    output_dir=str((MODELS_DIR / "yolov9c").resolve()),
    run_name="yolov9_full",
    project="nvd-car-detection",
)

full_best_pt = run_yolo_fine_tuning(full_config, aug_variant="full")
full_best_pt_dest = MODELS_DIR / "yolov9_full_best.pt"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(full_best_pt, full_best_pt_dest)

print(f"Full-aug checkpoint → {full_best_pt_dest}")

full_metrics = eval_yolo_checkpoint(
    weights=full_best_pt_dest,
    data_yaml=DATA_YAML,
    split="val",
    device=DEVICE,
)
append_comparison_row(full_metrics, COMPARISON_CSV)

print(f"\nmAP@0.5     : {full_metrics.map50:.4f}")
print(f"mAP@0.5:0.95: {full_metrics.map50_95:.4f}")
print(f"Precision   : {full_metrics.precision:.4f}")
print(f"Recall      : {full_metrics.recall:.4f}")
print(f"FPS         : {full_metrics.fps:.1f}")


### 4.10A RT-DETR Snow Augmentation Training

Fine-tune RT-DETR with the **snow pipeline**: `RandomSnow` + `RandomFog` + `RandomBrightnessContrast`.
Same architecture and hyperparameters as section 4.3 — augmentation is the only variable.
Checkpoint saved under `runs/rtdetr_snow/best_checkpoint`.


In [ ]:

# ── 4.10A RT-DETR snow augmentation training ───────────────────────────────────
rtdetr_snow_config = TrainingConfig(
    model_name="PekingU/rtdetr_r50vd",
    data_yaml="",
    epochs=100,
    batch=4,
    lr0=1e-5,
    freeze=None,
    device=DEVICE,
    output_dir=str((MODELS_DIR / "rtdetr").resolve()),
    run_name="rtdetr_snow",
)



rtdetr_snow_dir = run_detr_fine_tuning(
    rtdetr_snow_config,
    train_json=TRAIN_COCO_JSON,
    val_json=VAL_COCO_JSON,
    images_dir=IMAGES_DIR,
    aug_variant="snow",
)
print(f"RT-DETR snow checkpoint → {rtdetr_snow_dir}")

rtdetr_snow_metrics = eval_detr_checkpoint(
    checkpoint_dir=rtdetr_snow_dir,
    val_json=VAL_COCO_JSON,
    images_dir=IMAGES_DIR,
    split="val",
    device_str=DEVICE,
)
append_comparison_row(rtdetr_snow_metrics, COMPARISON_CSV)

print(f"\nmAP@0.5     : {rtdetr_snow_metrics.map50:.4f}")
print(f"mAP@0.5:0.95: {rtdetr_snow_metrics.map50_95:.4f}")
print(f"Precision   : {rtdetr_snow_metrics.precision:.4f}")
print(f"Recall      : {rtdetr_snow_metrics.recall:.4f}")
print(f"FPS         : {rtdetr_snow_metrics.fps:.1f}")


### 4.10B RT-DETR Full Augmentation Training

Add `GaussNoise` and `MotionBlur` on top of the snow pipeline.
Checkpoint saved under `runs/rtdetr_full/best_checkpoint`.


In [ ]:

# ── 4.10B RT-DETR full augmentation training ───────────────────────────────────
rtdetr_full_config = TrainingConfig(
    model_name="PekingU/rtdetr_r50vd",
    data_yaml="",
    epochs=100,
    batch=4,
    lr0=1e-5,
    freeze=None,
    device=DEVICE,
    output_dir=str((MODELS_DIR / "rtdetr").resolve()),
    run_name="rtdetr_full",
)



rtdetr_full_dir = run_detr_fine_tuning(
    rtdetr_full_config,
    train_json=TRAIN_COCO_JSON,
    val_json=VAL_COCO_JSON,
    images_dir=IMAGES_DIR,
    aug_variant="full",
)
print(f"RT-DETR full checkpoint → {rtdetr_full_dir}")

rtdetr_full_metrics = eval_detr_checkpoint(
    checkpoint_dir=rtdetr_full_dir,
    val_json=VAL_COCO_JSON,
    images_dir=IMAGES_DIR,
    split="val",
    device_str=DEVICE,
)
append_comparison_row(rtdetr_full_metrics, COMPARISON_CSV)

print(f"\nmAP@0.5     : {rtdetr_full_metrics.map50:.4f}")
print(f"mAP@0.5:0.95: {rtdetr_full_metrics.map50_95:.4f}")
print(f"Precision   : {rtdetr_full_metrics.precision:.4f}")
print(f"Recall      : {rtdetr_full_metrics.recall:.4f}")
print(f"FPS         : {rtdetr_full_metrics.fps:.1f}")


### 4.11A Faster R-CNN Snow Augmentation Training

Fine-tune Faster R-CNN with the **snow pipeline**: `RandomSnow` + `RandomFog` + `RandomBrightnessContrast`.
Same architecture and hyperparameters as section 4.4 — augmentation is the only variable.
Checkpoint saved to `models/frcnn_snow_best.pt`.


In [ ]:

# ── 4.11A Faster R-CNN snow augmentation training ──────────────────────────────
frcnn_snow_config = TrainingConfig(
    model_name="fasterrcnn_resnet50_fpn",
    data_yaml=DATA_YAML,
    epochs=100,
    batch=4,
    lr0=0.005,
    freeze=None,
    device=DEVICE,
    output_dir=str((MODELS_DIR / "frcnn").resolve()),
    run_name="frcnn_snow",
)



frcnn_snow_pt = run_frcnn_fine_tuning(
    frcnn_snow_config, data_yaml=DATA_YAML, aug_variant="snow"
)
frcnn_snow_dest = MODELS_DIR / "frcnn_snow_best.pt"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(frcnn_snow_pt, frcnn_snow_dest)
print(f"Faster R-CNN snow checkpoint → {frcnn_snow_dest}")

frcnn_snow_metrics = eval_frcnn_checkpoint(
    weights=frcnn_snow_dest,
    data_yaml=DATA_YAML,
    split="val",
    device_str=DEVICE,
)
append_comparison_row(frcnn_snow_metrics, COMPARISON_CSV)

print(f"\nmAP@0.5     : {frcnn_snow_metrics.map50:.4f}")
print(f"mAP@0.5:0.95: {frcnn_snow_metrics.map50_95:.4f}")
print(f"Precision   : {frcnn_snow_metrics.precision:.4f}")
print(f"Recall      : {frcnn_snow_metrics.recall:.4f}")
print(f"FPS         : {frcnn_snow_metrics.fps:.1f}")


### 4.11B Faster R-CNN Full Augmentation Training

Add `GaussNoise` and `MotionBlur` on top of the snow pipeline.
Checkpoint saved to `models/frcnn_full_best.pt`.


In [ ]:

# ── 4.11B Faster R-CNN full augmentation training ──────────────────────────────
frcnn_full_config = TrainingConfig(
    model_name="fasterrcnn_resnet50_fpn",
    data_yaml=DATA_YAML,
    epochs=100,
    batch=4,
    lr0=0.005,
    freeze=None,
    device=DEVICE,
    output_dir=str((MODELS_DIR / "frcnn").resolve()),
    run_name="frcnn_full",
)

frcnn_full_pt = run_frcnn_fine_tuning(
    frcnn_full_config, data_yaml=DATA_YAML, aug_variant="full"
)
frcnn_full_dest = MODELS_DIR / "frcnn_full_best.pt"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(frcnn_full_pt, frcnn_full_dest)
print(f"Faster R-CNN full checkpoint → {frcnn_full_dest}")

frcnn_full_metrics = eval_frcnn_checkpoint(
    weights=frcnn_full_dest,
    data_yaml=DATA_YAML,
    split="val",
    device_str=DEVICE,
)
append_comparison_row(frcnn_full_metrics, COMPARISON_CSV)

print(f"\nmAP@0.5     : {frcnn_full_metrics.map50:.4f}")
print(f"mAP@0.5:0.95: {frcnn_full_metrics.map50_95:.4f}")
print(f"Precision   : {frcnn_full_metrics.precision:.4f}")
print(f"Recall      : {frcnn_full_metrics.recall:.4f}")
print(f"FPS         : {frcnn_full_metrics.fps:.1f}")


### 4.12 Augmentation Ablation Results

Compare val-set mAP across **all 9 combinations** (3 models × 3 aug variants) and produce
a grouped bar chart.  Saves `results/figures/aug_ablation_bar.png` for the presentation slides.


In [ ]:

# ── 4.12 Augmentation ablation results — all 3 models × 3 variants ─────────────
aug_rows = []

# ── YOLOv9c ───────────────────────────────────────────────────────────────────
for label, rn in [
    ("YOLOv9c · no aug",   "yolov9_nvd"),
    ("YOLOv9c · snow aug", "yolov9_snow"),
    ("YOLOv9c · full aug", "yolov9_full"),
]:
    w = MODELS_DIR / f"{rn}_best.pt"
    if not w.exists():
        print(f"Skipping {label} — {w} not found"); continue
    m = eval_yolo_checkpoint(weights=w, data_yaml=DATA_YAML,
                             split="val", device=DEVICE, model_label=label)
    append_comparison_row(m, COMPARISON_CSV)
    aug_rows.append({"Model": label, "mAP@0.5": m.map50, "mAP@0.5:0.95": m.map50_95,
                     "Precision": m.precision, "Recall": m.recall, "FPS": m.fps})

# ── RT-DETR ───────────────────────────────────────────────────────────────────
for label, rn in [
    ("RT-DETR · no aug",   "rtdetr_nvd"),
    ("RT-DETR · snow aug", "rtdetr_snow"),
    ("RT-DETR · full aug", "rtdetr_full"),
]:
    d = (MODELS_DIR / "rtdetr").resolve() / rn / "best_checkpoint"
    if not d.exists():
        print(f"Skipping {label} — {d} not found"); continue
    m = eval_detr_checkpoint(checkpoint_dir=d, val_json=VAL_COCO_JSON,
                             images_dir=IMAGES_DIR, split="val", device_str=DEVICE)
    append_comparison_row(m, COMPARISON_CSV)
    aug_rows.append({"Model": label, "mAP@0.5": m.map50, "mAP@0.5:0.95": m.map50_95,
                     "Precision": m.precision, "Recall": m.recall, "FPS": m.fps})

# ── Faster R-CNN ──────────────────────────────────────────────────────────────
for label, rn in [
    ("Faster R-CNN · no aug",   "frcnn_nvd"),
    ("Faster R-CNN · snow aug", "frcnn_snow"),
    ("Faster R-CNN · full aug", "frcnn_full"),
]:
    w = MODELS_DIR / f"{rn}_best.pt"
    if not w.exists():
        print(f"Skipping {label} — {w} not found"); continue
    m = eval_frcnn_checkpoint(weights=w, data_yaml=DATA_YAML,
                              split="val", device_str=DEVICE)
    append_comparison_row(m, COMPARISON_CSV)
    aug_rows.append({"Model": label, "mAP@0.5": m.map50, "mAP@0.5:0.95": m.map50_95,
                     "Precision": m.precision, "Recall": m.recall, "FPS": m.fps})

# ── Display table ─────────────────────────────────────────────────────────────
if aug_rows:
    df_aug  = pd.DataFrame(aug_rows)
    numeric = ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "FPS"]
    display(
        df_aug.style
          .highlight_max(subset=numeric, color="lightgreen")
          .format({c: "{:.4f}" for c in numeric if c != "FPS"})
          .format({"FPS": "{:.1f}"})
          .set_caption("4.9 — Augmentation Ablation (val set, all models)")
    )

    # ── Grouped bar chart ─────────────────────────────────────────────────────
    _model_prefixes = ["YOLOv9c", "RT-DETR", "Faster R-CNN"]
    _variants       = ["no aug", "snow aug", "full aug"]
    _colors         = ["steelblue", "orange", "green"]
    x = np.arange(len(_model_prefixes))
    bar_w = 0.25

    fig, ax = plt.subplots(figsize=(9, 5))
    for i, (variant, color) in enumerate(zip(_variants, _colors)):
        vals = [
            next((r["mAP@0.5"] for r in aug_rows
                  if r["Model"].startswith(mp) and variant in r["Model"]), 0.0)
            for mp in _model_prefixes
        ]
        bars = ax.bar(x + (i - 1) * bar_w, vals, bar_w,
                      label=variant, color=color, alpha=0.85)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, v + 0.005,
                    f"{v:.3f}", ha="center", fontsize=7.5)

    ax.set_xticks(x)
    ax.set_xticklabels(_model_prefixes)
    ax.set_ylabel("mAP@0.5")
    ax.set_ylim(0, 1)
    ax.set_title("Augmentation Ablation — All Models (val set)")
    ax.legend(title="Augmentation")
    plt.tight_layout()
    fig.savefig(FIGURES / "aug_ablation_bar.png", dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Saved → {FIGURES / 'aug_ablation_bar.png'}")


### 4.13 Hyperparameter Sweep

Grid search over `lr0` and `imgsz` using W&B Sweeps.
Each trial trains for 30 epochs on the standard (no-aug) data so results are comparable.
Pick the best `lr0` / `imgsz` combination and plug it into section 4.11.


In [ ]:

# ── 4.13 Hyperparameter sweep with W&B ────────────────────────────────────────
# Grid: lr0 × imgsz  (3 × 2 = 6 runs; extend by adding epochs values if compute allows)
import wandb

sweep_config = {
    "method": "grid",
    "name": "yolov9-hparam-sweep",
    "metric": {"name": "val/map50", "goal": "maximize"},
    "parameters": {
        "lr0":   {"values": [0.001, 0.005, 0.01]},
        "imgsz": {"values": [416, 640]},
    },
}


def _sweep_run():
    with wandb.init() as run:
        cfg = run.config
        config = TrainingConfig(
            model_name="yolov9c",
            data_yaml=DATA_YAML,
            epochs=30,          # shorter runs during sweep
            batch=16,
            imgsz=cfg.imgsz,
            lr0=cfg.lr0,
            freeze=None,
            device=DEVICE,
            output_dir=str((ROOT / "runs").resolve()),
            run_name=f"sweep_{run.id}",
            project="nvd-car-detection",
        )
        best_pt  = run_yolo_fine_tuning(config)
        metrics  = eval_yolo_checkpoint(
            weights=best_pt, data_yaml=DATA_YAML,
            split="val", device=DEVICE,
            model_label=f"sweep lr={cfg.lr0} imgsz={cfg.imgsz}",
        )
        wandb.log({"val/map50": metrics.map50, "val/map50_95": metrics.map50_95,
                   "val/precision": metrics.precision, "val/recall": metrics.recall})


sweep_id = wandb.sweep(sweep_config, project="nvd-car-detection")
# count=6 runs all 6 grid combinations; reduce for a quick sanity-check
wandb.agent(sweep_id, _sweep_run, count=6)

print("Sweep complete. View results at:")
print(f"  https://wandb.ai/your-team/nvd-car-detection/sweeps/{sweep_id}")


### 4.14 Final Model Training

Train the final production model using the best augmentation variant (full) and the best
hyperparameters identified in the sweep.  This checkpoint is saved to `models/final_best_model.pt`
and used for all test-set evaluation in Section 5.


In [ ]:

# ── 4.14 Final model — best aug + best hyperparams from sweep ─────────────────
# Update lr0 / imgsz below with the best values found in section 4.10.
import shutil

final_config = TrainingConfig(
    model_name="yolov9c",
    data_yaml=DATA_YAML,
    epochs=100,
    batch=16,
    imgsz=1024,
    lr0=0.005,      # ← update from sweep results
    freeze=None,
    device=DEVICE,
    output_dir=str((MODELS_DIR / "final_model").resolve()),
    run_name="yolov9_final",
    project="nvd-car-detection",
)

_final_best_pt = run_yolo_fine_tuning(final_config, aug_variant="full")

final_dest = MODELS_DIR / "final_best_model.pt"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(_final_best_pt, final_dest)
print(f"Final model → {final_dest}")

final_metrics = eval_yolo_checkpoint(
    weights=final_dest,
    data_yaml=DATA_YAML,
    split="val",
    device=DEVICE,
    model_label="YOLOv9c (final)",
)
append_comparison_row(final_metrics, COMPARISON_CSV)
print(f"\nmAP@0.5     : {final_metrics.map50:.4f}")
print(f"mAP@0.5:0.95: {final_metrics.map50_95:.4f}")
print(f"Precision   : {final_metrics.precision:.4f}")
print(f"Recall      : {final_metrics.recall:.4f}")
print(f"FPS         : {final_metrics.fps:.1f}")


## 5. Evaluation

In [ ]:
# TODO: load best checkpoint and evaluate on the held-out test set
# model = YOLO(str(MODELS_DIR / "final_best_model.pt"))
# metrics = model.val(data="configs/data.yaml", split="test")
# print(metrics)
pass

## 6. Error Analysis

In [ ]:
# TODO: identify ~20 false negatives, ~10 false positives, ~10 poor localisation cases
# TODO: visualise failure grids (3x3 image grids per category)
pass